In [1]:
import json
from pathlib import Path

import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

In [2]:
#ucitavanje podataka


In [3]:
PROJECT_ROOT = Path.cwd()

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)
TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.jsonl"
)
VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "validation.jsonl"
)
TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.jsonl"
)

In [4]:
def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                records.append(record)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records

In [5]:
chunks = load_jsonl(CHUNKS_PATH)

train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

In [6]:
#provera 
print(f"Broj chunkova: {len(chunks)}")
print(f"Training pitanja: {len(train_data)}")
print(f"Validation pitanja: {len(validation_data)}")
print(f"Test pitanja: {len(test_data)}")

Broj chunkova: 344
Training pitanja: 100
Validation pitanja: 21
Test pitanja: 22


In [7]:
#ucitavanje modela

In [8]:
RETRIEVER_MODEL_NAME = "intfloat/multilingual-e5-base"

retriever_model = SentenceTransformer(
    RETRIEVER_MODEL_NAME
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
#provera
test_embedding = retriever_model.encode(
    ["query: Zašto je bitan kvalitet softvera?"],
    normalize_embeddings=True
)

print(test_embedding.shape)

(1, 768)


In [10]:
passages = [
    "passage: " + chunk["processed_text"]
    for chunk in chunks
]

print(f"Pripremljeno tekstova: {len(passages)}")

Pripremljeno tekstova: 344


In [11]:
chunk_embeddings = retriever_model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [12]:
#provera
print(f"Oblik matrice embeddinga: {chunk_embeddings.shape}")
print(f"Tip podataka: {chunk_embeddings.dtype}")

Oblik matrice embeddinga: (344, 768)
Tip podataka: float32


In [13]:
chunk_embeddings = chunk_embeddings.astype("float32")

embedding_dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(chunk_embeddings)

In [14]:
#provera
print(f"Broj vektora u indeksu: {faiss_index.ntotal}")

Broj vektora u indeksu: 344


In [15]:
def retrieve(
    question: str,
    top_k: int = 5
) -> list[dict]:

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Pitanje ne sme biti prazno.")

    query = "query: " + question.strip()

    query_embedding = retriever_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    number_of_results = min(
        top_k,
        len(chunks)
    )

    scores, indices = faiss_index.search(
        query_embedding,
        number_of_results
    )

    results = []

    for rank, (score, chunk_index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        chunk = chunks[int(chunk_index)]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "score": float(score),
            "text": chunk["processed_text"],
            "pdf_page_start": chunk["pdf_page_start"],
            "pdf_page_end": chunk["pdf_page_end"],
            "section_ref": chunk.get("section_ref"),
            "heading": chunk.get("heading")
        })

    return results

In [16]:
#provera na jednom primeru
QUESTION_ID = 61

example = next(
    example
    for example in test_data
    if example["id"] == QUESTION_ID
)

question = example["processed_question"]
print("Pitanje:")
print(question)
print("Odgovor:")
print(example["processed_answer"])
retrieved_chunks = retrieve(
    question=question,
    top_k=5
)

Pitanje:
Opisati testove kompatibilnosti pri testiranju softvera.
Odgovor:
Testovi kompatibilnosti proveravaju da softver može pravilno da radi u različitim okruženjima i zajedno sa drugim programima, uređajima ili servisima.


In [17]:
for result in retrieved_chunks:
    print("\n" + "=" * 80)

    print(f'Rang: {result["rank"]}')
    print(f'Chunk: {result["chunk_id"]}')
    print(f'Sličnost: {result["score"]:.4f}')

    print(
        "PDF stranice:",
        result["pdf_page_start"],
        "-",
        result["pdf_page_end"]
    )
    print("\nTekst:")
    print(result["text"][:700])


Rang: 1
Chunk: chunk_0167
Sličnost: 0.8609
PDF stranice: 97 - 98

Tekst:
[Vrste i nivoi testiranja]

čkih podešavanja tokom aktivne sesije — provera se uticaj na postojeći sadržaj korisničkog interfejsa.

U svakom od prethodnih situacija aplikacija bi trebalo da se ponaša stabilno, obezbedi korisniku jasne poruke o greškama i izbegne pad sistema ili gubitak podataka. Cilj ovih provera je otkrivanje potencijalnih grešaka i nepredviđenih ponašanja sistema u realnim, kompleksnim i neobičnim scenarijima koje standardni test slučajevi možda ne pokrivaju.

Testiranje prihvatljivosti

Testovi prihvatljivosti (eng. acceptance testing) treba da omoguće klijentima i korisnicima da se sami uvere da je napravljeni softver u skladu sa njihovim potrebama i očekivanjima. Ovu vr

Rang: 2
Chunk: chunk_0172
Sličnost: 0.8539
PDF stranice: 100 - 100

Tekst:
[Testiranje > 4.3 Tehnike testiranja]

sa krajnjim korisnicima, kako bi se osiguralo da instalacija teče bez grešaka i da sistem nakon instalacije is